In [1]:
import onnxruntime as ort
import uniface # NEW: Import uniface

# Existing detector
session = ort.InferenceSession(
    "face_detector.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_names = [o.name for o in session.get_outputs()]


In [3]:
from flask import Flask, request, jsonify
import cv2
import numpy as np
import onnxruntime as ort
import requests # NEW: Import requests
import uniface  # NEW: Import uniface

app = Flask(__name__)

# --- CONFIGURATION ---
EXTERNAL_API_BASE_URL = "http://localhost:8080" # PLACEHOLDER: Change to your actual REST server URL
# ---------------------

# Cargar modelo detection
session = ort.InferenceSession(
    "face_detector.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_names = [o.name for o in session.get_outputs()]

# Initialize Recognition (ArcFace is default/common)
# Assuming uniface has a simple interface, otherwise we might need to adjust initialization
recognizer = uniface.FaceRecognition('arcface')

def preprocess_image(image):
    # 1. RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape
    
    # 2. Letterbox
    target_size = 224
    scale = min(target_size / w, target_size / h)
    nw = int(w * scale)
    nh = int(h * scale)
    image_resized = cv2.resize(image, (nw, nh))
    
    new_image = np.full((target_size, target_size, 3), 128, dtype=np.uint8)
    dx = (target_size - nw) // 2
    dy = (target_size - nh) // 2
    new_image[dy:dy+nh, dx:dx+nw] = image_resized
    
    # 3. Normalize
    new_image = new_image.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    new_image = (new_image - mean) / std
    
    # 4. Transpose
    new_image = np.transpose(new_image, (2, 0, 1))
    new_image = np.expand_dims(new_image, axis=0)
    
    return new_image.astype(np.float32), (scale, dx, dy)

@app.route("/detect", methods=["POST"])
def detect():
    if "image" not in request.files:
        return jsonify({"error": "No image provided"}), 400

    # Get Group ID from Query Param
    group_id = request.args.get("group_id", "default_group")

    file = request.files["image"]
    image_bytes = np.frombuffer(file.read(), np.uint8)
    image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)
    original_image = image.copy()

    # --- Preprocessing ---
    input_tensor, (scale, dx, dy) = preprocess_image(image)

    # --- Inference ---
    outputs = session.run(
        output_names,
        {input_name: input_tensor}
    )

    # --- Postprocessing ---
    score = outputs[0][0][0]
    bbox = outputs[1][0] # [x, y, w, h] normalized to 224x224 canvas
    
    detections = [float(score)]
    box_coords = []
    identity_info = {}

    if score > 0.5:
        t_size = 224
        x_c, y_c, w_c, h_c = bbox # coords on canvas (0-1)
        
        # 1. Denormalize to canvas pixels
        x_px_c = x_c * t_size
        y_px_c = y_c * t_size
        w_px_c = w_c * t_size
        h_px_c = h_c * t_size
        
        # 2. Remove padding (dx, dy)
        x_no_pad = x_px_c - dx
        y_no_pad = y_px_c - dy
        
        # 3. Rescale to original size
        x_final = int(x_no_pad / scale)
        y_final = int(y_no_pad / scale)
        w_final = int(w_px_c / scale)
        h_final = int(h_px_c / scale)
        
        # --- HEURISTIC: EXPAND BOX 50% ---
        margin_w = int(w_final * 0.5)
        margin_h = int(h_final * 0.5)
        
        x_final = max(0, x_final - margin_w // 2)
        y_final = max(0, y_final - margin_h // 2)
        h_img, w_img, _ = image.shape
        w_final = min(w_img - x_final, w_final + margin_w)
        h_final = min(h_img - y_final, h_final + margin_h)
        # ---------------------------------

        # --- RECOGNITION (Uniface) ---
        try:
            # Crop face
            face_crop = original_image[y_final:y_final+h_final, x_final:x_final+w_final]
            
            if face_crop.size > 0:
                # Get Embedding (Assuming uniface.represent or similar)
                # Adjust this method call based on specific uniface API if needed
                embedding = recognizer.represent(face_crop)
                
                # Call External API
                external_url = f"{EXTERNAL_API_BASE_URL}/group/{group_id}/recognize"
                print(f"Calling External API: {external_url}")
                
                # Sending embedding as list
                ext_response = requests.post(external_url, json={"embedding": embedding.tolist()})
                
                if ext_response.status_code == 200:
                    identity_info = ext_response.json()
                else:
                    identity_info = {"error": f"External API Error: {ext_response.status_code}", "details": ext_response.text}
            else:
                 identity_info = {"error": "Invalid Face Crop"}

        except Exception as e:
            print(f"Recognition Error: {e}")
            identity_info = {"error": str(e)}
        # -----------------------------

        cv2.rectangle(image, (x_final, y_final), (x_final+w_final, y_final+h_final), (0, 0, 255), 2)
        cv2.putText(image, f"{score:.2f}", (x_final, y_final-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
        
        box_coords = [x_final, y_final, w_final, h_final]
        
    cv2.imwrite("detected.jpg", image)

    response_data = {
        "detections": detections,
        "box": box_coords,
        "identity": identity_info
    }
    return jsonify(response_data)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)


AttributeError: module 'uniface' has no attribute 'FaceRecognition'